<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>Hugging Face 라이브러리(Hugging Face Transformers)</strong>에 대해 학습합니다.</br></br>
>AutoTokenizer, AutoModel, pipeline API로 사전학습 모델을 학습해봅시다.

<div style="text-align:center">

</div>

</br>

# Hugging Face Transformers
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">사전학습된 NLP 모델을 쉽게 사용</mark>할 수 있게 해주는 라이브러리입니다.

> 처음부터 모델을 학습하는 것은 현실적으로 불가능합니다. BERT-base 학습에는 TPU v3 64개로 4일, GPT-3 학습에는 수백만 달러의 비용이 들었습니다. 따라서 대규모 텍스트 코퍼스로 범용 언어 표현을 미리 학습(Pre-training)한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">공개된 사전 학습 모델을 재사용</mark>하는 것이 유일한 현실적 선택입니다.</br></br>
> 이렇게 학습된 모델은 문법, 상식, 사실 지식까지 내재화되어 있어, 소량의 태스크 데이터로 파인튜닝(Fine-tuning)만 해도 월등한 성능을 얻을 수 있습니다.</br></br>
> Hugging Face가 사실상 표준이 된 이유는 생태계의 통일 덕분입니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Model Hub</mark>에서 BERT, GPT, T5, LLaMA 등 수십만 개의 체크포인트(학습된 가중치 파일)를 다운로드할 수 있고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Auto 클래스</mark>로 모델명 하나만 지정하면 토크나이저, 모델, 설정이 자동으로 로드됩니다.</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">pipeline API</mark>를 사용하면 단 한 줄로 감성 분석, 번역, 생성 등을 실행할 수 있으며, 논문 저자들이 직접 가중치를 업로드하여 재현성까지 보장됩니다.</br></br>
> 결과적으로 Hugging Face는 NLP 연구와 실무 모두에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">공통 언어</mark>가 되었습니다.

</br>

## AutoTokenizer
> 모델에 맞는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토크나이저를 자동으로 선택</mark>하고 로드합니다.

In [11]:
# TODO 1: 사전학습된 토크나이저를 불러와 "Hello, world!"를 인코딩하여 input_ids, attention_mask, 토큰 목록, 디코딩 결과를 출력해봅시다.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

encoded = tokenizer("Hello, world!", return_tensors="pt")
print(f"input_ids:      {encoded['input_ids']}")
print(f"attention_mask: {encoded['attention_mask']}")
print(f"토큰 목록:       {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")
print(f"디코딩 결과:     {tokenizer.decode(encoded['input_ids'][0])}")

input_ids:      tensor([[ 101, 7592, 1010, 2088,  999,  102]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1]])
토큰 목록:       ['[CLS]', 'hello', ',', 'world', '!', '[SEP]']
디코딩 결과:     [CLS] hello, world! [SEP]


</br>

## AutoModel

In [12]:
# TODO 2: 사전학습된 시퀀스 분류 모델을 불러와 모델 타입과 파라미터 수를 출력해봅시다.

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"  # SST-2 감성 분류 학습 완료 모델
)
print(f"모델 타입: {type(model).__name__}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"레이블 매핑: {model.config.id2label}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 12486.62it/s]

모델 타입: DistilBertForSequenceClassification
파라미터 수: 66,955,010
레이블 매핑: {0: 'NEGATIVE', 1: 'POSITIVE'}


<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">Auto 클래스</th>
      <th>용도</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>AutoModel</code></td><td>기본 특성 추출</td></tr>
    <tr><td style="text-align:center"><code>AutoModelForSequenceClassification</code></td><td>텍스트 분류</td></tr>
    <tr><td style="text-align:center"><code>AutoModelForTokenClassification</code></td><td>개체명 인식 (NER)</td></tr>
    <tr><td style="text-align:center"><code>AutoModelForCausalLM</code></td><td>텍스트 생성</td></tr>
  </tbody>
</table>

</br>

## Pipeline API
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">한 줄로 추론</mark>을 수행하는 고수준 API입니다.

In [13]:
# TODO 3: 감성 분석 파이프라인을 생성하고, "I love this movie!"를 입력하여 결과를 출력해봅시다.

from transformers import pipeline

classifier = pipeline("sentiment-analysis")
result = classifier("I love this movie!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 15491.98it/s]


[{'label': 'POSITIVE', 'score': 0.9998775720596313}]


💡pipeline 태스크 종류
> `sentiment-analysis`, `translation`, `text-generation`, `summarization`,</br>
> `question-answering`, `fill-mask`, `ner`, `zero-shot-classification` 등

</br>

## 추론 패턴 (Manual Pipeline)

In [14]:
# TODO 4: 토크나이저로 "This is great!"를 인코딩한 뒤, 추론 모드에서 모델에 입력하여 로짓, 소프트맥스 확률, 예측 클래스를 출력해봅시다.

import torch

inputs = tokenizer("This is great!", return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits                        # 모델의 원시 출력 (학습되지 않은 점수)
probs = torch.softmax(logits, dim=-1)          # softmax로 확률 변환 (합=1)
predicted = torch.argmax(logits, dim=-1)       # 가장 높은 확률의 클래스 ID

label_name = model.config.id2label[predicted.item()]  # 클래스 ID → 이름

print(f"Logits:    {logits.data.round(decimals=3)}")
print(f"확률:      {probs.data.round(decimals=3)}")
print(f"예측 결과: {predicted.item()} → {label_name}")

Logits:    tensor([[-4.3030,  4.6410]])
확률:      tensor([[0., 1.]])
예측 결과: 1 → POSITIVE


💡pipeline vs 수동 추론
> pipeline: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">빠른 프로토타이핑</mark>, 기본 설정으로 충분할 때</br>
> 수동 추론: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">커스텀 후처리</mark>, 배치 처리, 성능 최적화가 필요할 때